# 22. Wave Strategy — Quản Lý Nhiều Tính Năng Cùng Lúc

**Phần 5: Thực hành từ đầu đến cuối**

---

Bạn đã biết cách xây 1 tính năng (bài 20) và sửa lỗi (bài 21). Nhưng dự án thật không chỉ có 1 tính năng — bạn cần xây 5, 10, hay 20 tính năng.

Làm sao quản lý tất cả mà không bị rối?

Hãy nghĩ về **đợt ship hàng**: thay vì gửi từng hộp lẻ (tốn công, khó kiểm soát), bạn gom nhiều hộp thành 1 đợt, kiểm tra kỹ rồi gửi 1 lần. An toàn hơn, hiệu quả hơn.

Đó chính là **Wave Strategy**.

## Wave Là Gì?

**Wave = 1 đợt công việc gồm nhiều PR (Pull Request)**

| Khái niệm | Ví von | Ví dụ |
|-----------|--------|-------|
| PR | 1 đơn hàng | "Thêm trang Landing Page" |
| Wave | 1 đợt ship gồm nhiều đơn | "Wave 1: Landing + Products + Auth" |
| Main | Kho hàng chính | Code ổn định, đã kiểm tra |

### Tại sao không merge từng PR lẻ?

Merge từng PR = gửi từng hộp. Có thể:
- Hộp A riêng lẻ thì OK
- Hộp B riêng lẻ thì OK
- Nhưng A + B chung thì va vào nhau!

Wave giúp bạn **kiểm tra A + B cùng nhau** trước khi đưa vào kho chính.

## Khi Nào Dùng Wave?

**Dùng Wave khi:**
- Có 3+ PR liên quan đến nhau
- Nhiều tính năng cần hoạt động cùng nhau
- Muốn kiểm tra tổng thể trước khi merge

**Không cần Wave khi:**
- Chỉ có 1 PR đơn lẻ, không ảnh hưởng gì khác
- Fix bug nhỏ, độc lập

Ví dụ thực tế:
```
Wave 1: MVP (Minimum Viable Product)
  - PR #1: Landing page
  - PR #2: Product listing page
  - PR #3: Contact form
  - PR #4: Authentication (đăng nhập/đăng ký)

Wave 2: Payment & Cart
  - PR #5: Shopping cart
  - PR #6: Checkout page
  - PR #7: Payment integration
```

## Wave Workflow — 7 Bước

### Bước 1: Plan Wave

Quyết định wave này gồm những PR nào. Nói với Claude:

```
Tôi muốn plan wave 1 cho dự án coffee shop.
Gồm: landing page, product listing, contact form.
Hãy giúp tôi chia task và estimate thời gian.
```

In [ ]:
# Buoc 2: Tao wave branch

cat << 'EOF'
=== TAO WAVE BRANCH ===

# Tao branch cho wave 1
$ git checkout -b wave/1

# Kiem tra ban dang o branch nao
$ git branch
  main
* wave/1

=> Ban dang o wave/1. Moi thay doi se nam o day,
   KHONG anh huong den main (kho chinh).
EOF

In [ ]:
# Vi du tao wave branch that
# (Trong du an that, ban chay lenh nay)

echo "=== Cac lenh tao wave ==="
echo ""
echo "git checkout main              # Ve main truoc"
echo "git pull                       # Lay code moi nhat"
echo "git checkout -b wave/1         # Tao wave branch"
echo ""
echo "=> Bay gio moi PR se duoc merge vao wave/1 truoc."
echo "   Khi wave xong, wave/1 moi duoc merge vao main."

### Bước 3-4: Agents Làm Song Song

Đây là phần hay nhất — bạn có thể có nhiều "phiên Claude" làm việc song song, mỗi phiên làm 1 PR.

```
Agent 1: Landing page     (feature/landing-page)
Agent 2: Product listing  (feature/product-listing)
Agent 3: Contact form     (feature/contact-form)
```

Mỗi agent làm trên branch riêng (worktree isolation) nên không ảnh hưởng lẫn nhau.

Khi từng agent xong, merge PR vào `wave/1`.

In [ ]:
# Moi Agent tao branch rieng tu wave/1:

cat << 'EOF'
=== PARALLEL WORK ===

# Agent 1:
$ git checkout -b feature/landing-page wave/1
  ... lam landing page ...
$ git push -u origin feature/landing-page
$ gh pr create --base wave/1    # PR vao wave, KHONG vao main!

# Agent 2 (song song):
$ git checkout -b feature/product-listing wave/1
  ... lam product listing ...
$ git push -u origin feature/product-listing
$ gh pr create --base wave/1

# Agent 3 (song song):
$ git checkout -b feature/contact-form wave/1
  ... lam contact form ...
$ git push -u origin feature/contact-form
$ gh pr create --base wave/1

=> 3 PR duoc merge vao wave/1.
   Main van chua bi anh huong.
EOF

### Bước 5: Quality Check Trước Khi Merge Wave

Đây là bước QUAN TRỌNG NHẤT. Trước khi đưa wave vào main, kiểm tra kỹ.

Bạn nói với Claude:
```
/quality-audit
```

Và:
```
/business-gap-check
```

In [ ]:
cat << 'EOF'
=== QUALITY CHECK KET QUA (vi du) ===

Quality Audit: 87/100 (B+)  => DAT

Business Gap Check:
  P1 (critical): 0  => DAT
  P2 (major):    1  => "Product page chua co sort/filter"
  P3 (minor):    2  => "Footer chua co social media links"

=> P1 = 0, co the merge!
   P2, P3 se lam o wave tiep theo.
EOF

### Bước 6-7: Tạo PR Wave → Main và Merge

In [ ]:
# Tao PR tu wave/1 vao main

cat << 'EOF'
=== MERGE WAVE VAO MAIN ===

# Day wave len GitHub
$ git checkout wave/1
$ git push -u origin wave/1

# Tao PR: wave/1 -> main
$ gh pr create --base main --title "Wave 1: MVP - Landing, Products, Contact" \
    --body "## Wave 1 Summary
    - Landing page with hero and features
    - Product listing with images
    - Contact form with validation
    
    Quality: 87/100 (B+)
    Business Gap: 0 P1"

# Merge (squash de gom tat ca commits thanh 1)
$ gh pr merge <so-PR> --squash --delete-branch

=> Wave 1 da vao main! San pham da co 3 trang!
EOF

## Wave Completion Check — 6 Levels

Trước khi merge wave, cần qua 6 cấp kiểm tra:

| Level | Tên | Kiểm tra gì | Ví von |
|-------|-----|-------------|--------|
| 1 | CI Green | Tất cả test pass | Kiểm tra xe có nổ máy không |
| 2 | Integration | Các module chạy cùng nhau | Kiểm tra các bộ phận xe ăn khớp |
| 3 | Business Logic | Nghiệp vụ đúng | Xe chạy đúng hướng |
| 4 | Regression | Không hỏng tính năng cũ | Bánh xe cũ vẫn quay |
| 5 | Documentation | Tài liệu cập nhật | Sổ bảo hành ghi đầy đủ |
| 6 | Post-wave Docs | Ghi nhận kết quả | Biên bản bàn giao |

Claude sẽ chạy qua 6 level này khi bạn gọi `/wave-completion-check`.

In [ ]:
cat << 'EOF'
=== WAVE COMPLETION CHECK (vi du) ===

Level 1 - CI Green:           PASS (24/24 tests pass)
Level 2 - Integration:        PASS (3 pages hoat dong cung nhau)
Level 3 - Business Logic:     PASS (products hien thi dung gia)
Level 4 - Regression:         PASS (khong co tinh nang cu bi hong)
Level 5 - Documentation:      PASS (README + business docs da update)
Level 6 - Post-wave Docs:     PASS (wave summary da ghi)

RESULT: 6/6 PASS => San sang merge!
EOF

## Vai Trò Của Bạn Trong Wave

Với Wave Strategy, vai trò của bạn rõ ràng hơn bao giờ hết:

| Bạn (Product Owner) | Claude (Developer) | Kit (QA Team) |
|--------------------|--------------------|---------------|
| Quyết định wave gồm gì | Implement từng PR | Quality audit |
| Ưu tiên tính năng nào trước | Viết code + tests | Business gap check |
| Review kết quả quality check | Git workflow | Wave completion check |
| Approve/reject merge | Fix lỗi | CI/CD pipeline |

Bạn là người **ra quyết định**, không phải người viết code.

## Ví Dụ Plan 3 Waves Cho Coffee Shop App

```
Wave 1: MVP (2-3 ngay)
  - Landing page
  - Product listing
  - Contact form
  => Muc tieu: co trang web co ban de gioi thieu

Wave 2: E-Commerce (3-5 ngay)
  - Shopping cart
  - Checkout + Payment
  - Order history
  => Muc tieu: khach hang co the mua hang

Wave 3: Enhancement (2-3 ngay)
  - User reviews
  - Search + Filter
  - Email notifications
  => Muc tieu: trai nghiem tot hon
```

Mỗi wave kết thúc = sản phẩm ổn định hơn, nhiều tính năng hơn.

In [ ]:
# Tong hop cac lenh Git cho Wave workflow

cat << 'EOF'
=== WAVE WORKFLOW — QUICK REFERENCE ===

# 1. Tao wave
git checkout main && git pull
git checkout -b wave/1

# 2. Tao feature branch trong wave
git checkout -b feature/landing-page wave/1

# 3. Lam xong feature -> PR vao wave
git push -u origin feature/landing-page
gh pr create --base wave/1
gh pr merge <N> --squash --delete-branch

# 4. Lam tiep feature khac (lap lai buoc 2-3)

# 5. Quality check tren wave/1
# /quality-audit
# /wave-completion-check

# 6. Merge wave vao main
git push -u origin wave/1
gh pr create --base main --title "Wave 1: MVP"
gh pr merge <N> --squash --delete-branch

# 7. Bat dau wave tiep theo
git checkout main && git pull
git checkout -b wave/2
EOF

## Tóm Tắt

- **Wave** = nhóm nhiều PR thành 1 đợt, kiểm tra kỹ rồi mới merge
- **Lợi ích**: phát hiện lỗi tích hợp, kiểm soát chất lượng tốt hơn
- **6 levels** kiểm tra: CI, Integration, Business, Regression, Docs, Post-wave
- **Bạn plan**, Claude implement, Kit kiểm tra

Bài tiếp theo — bài cuối cùng: Ship sản phẩm ra thế giới!